In [ ]:
# ! pip install -U kaleido


   -------------------------- ------------- 4/6 [choreographer]
   ---------------------------------------- 6/6 [kaleido]



In [46]:
import requests
from xml.etree import ElementTree
import json
import pprint
import time
import aiohttp
import asyncio
import tqdm
from sklearn import decomposition
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

In [26]:
# installed PyTorch
# ! pip install torch torchvision

In [27]:
# insured torch installed correctly
# asked ChatGPT how to guarentee it worked
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.9.0+cpu
CUDA available: False


In [28]:
# installed huggingfacefrom transformers import AutoTokenizer, AutoModel

# ! pip install transformers

In [29]:
from transformers import AutoTokenizer, AutoModel

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
model = AutoModel.from_pretrained('allenai/specter')

In [ ]:
# pull in 1000 alz articles
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed",
    "term": "Alzheimers AND 2024[pdat]",
    "retmax": "1000",
    "retmode": "xml"
}

response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

alz_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(alz_ids)} Alzheimers articles.")

Found 1000 Alzheimers articles.


In [31]:
# pull in 1000 cancer paper

base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed", # from PubMed
    "term": "cancer AND 2024[pdat]", #cancer
    "retmax": "1000", # only send 1000 articles
    "retmode": "xml" # sends in xml format
}

response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

cancer_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(cancer_ids)} cancer articles.")

Found 1000 cancer articles.


In [ ]:
# asked ChatGPT how to now get metadata for each article ID fetched earlier
# used ChatGPT to add the time.sleep portion, add article id title being pulled for metadata

# fetch metadata for the 1000 alzheimers
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
all_alz_metadata = []
failed_pmids = []
alz_pmids =[]

for i in range(0, len(alz_ids), 200):  # batches of 200
    batch_ids = ",".join(alz_ids[i:i+200])
    fetch_params = {
        "db": "pubmed",
        "id": batch_ids,
        "retmode": "xml"
    }
    try:
        fetch_response = requests.get(fetch_url, params=fetch_params, timeout=30)
        fetch_response.raise_for_status()
        root = ElementTree.fromstring(fetch_response.text)
    except Exception as e:
        print(f"Error fetching batch {i//200+1}: {e}")
        continue

    for article in root:
        try:
            title_elem = article.find(".//ArticleTitle")
            abstract_elems = article.findall(".//Abstract/AbstractText")
            journal_elem = article.find(".//Journal/Title")
            pmid_elem = article.find(".//PMID")
            date_elem = article.find(".//PubDate/Year")

            title_text = (
                ElementTree.tostring(title_elem, method="text", encoding="unicode").strip()
                if title_elem is not None
                else None
            )
            abstract_text = (
                " ".join(
                    ElementTree.tostring(elem, method="text", encoding="unicode").strip()
                    if elem.get("Label") else ElementTree.tostring(elem, method="text", encoding="unicode").strip()
                    for elem in abstract_elems
            )
            if abstract_elems
            else None
            )

            metadata = {
                "PMID": pmid_elem.text if pmid_elem is not None else None,
                "ArticleTitle": title_text,
                "AbstractText": abstract_text,
                "Journal": journal_elem.text if journal_elem is not None else None,
                "YearPublished": date_elem.text if date_elem is not None else None
            }
            all_alz_metadata.append(metadata)
            alz_pmids.append(metadata['PMID'])
        except Exception as e:
            pmid_text = pmid_elem.text if pmid_elem is not None else "UNKNOWN"
            print(f"Error parsing article PMID {pmid_text}: {e}")
            failed_pmids.append(pmid_text)
            continue

    time.sleep(1)

print(f"Retrieved metadata for {len(all_alz_metadata)} Alzheimers articles.")
print(f"Failed PMIDs in first pass: {len(failed_pmids)}")


Retrieved metadata for 1000 Alzheimers articles.
Failed PMIDs in first pass: 0


In [ ]:
# asked ChatGPT how to now get metadata for each article ID fetched earlier
# used ChatGPT to add the time.sleep portion, add article id title being pulled for metadata

# fetch metadata for the 1000 cancer
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
all_cancer_metadata = []
failed_pmids = []
cancer_pmids =[]

for i in range(0, len(cancer_ids), 200):  # batches of 200
    batch_ids = ",".join(cancer_ids[i:i+200])
    fetch_params = {
        "db": "pubmed",
        "id": batch_ids,
        "retmode": "xml"
    }
    try:
        fetch_response = requests.get(fetch_url, params=fetch_params, timeout=30)
        fetch_response.raise_for_status()
        root = ElementTree.fromstring(fetch_response.text)
    except Exception as e:
        print(f"Error fetching batch {i//200+1}: {e}")
        continue

    for article in root:
        try:
            title_elem = article.find(".//ArticleTitle")
            abstract_elems = article.findall(".//Abstract/AbstractText")
            journal_elem = article.find(".//Journal/Title")
            pmid_elem = article.find(".//PMID")
            date_elem = article.find(".//PubDate/Year")

            title_text = (
                ElementTree.tostring(title_elem, method="text", encoding="unicode").strip()
                if title_elem is not None
                else None
            )
            abstract_text = (
                " ".join(
                    ElementTree.tostring(elem, method="text", encoding="unicode").strip()
                    if elem.get("Label") else ElementTree.tostring(elem, method="text", encoding="unicode").strip()
                    for elem in abstract_elems
            )
            if abstract_elems
            else None
            )

            metadata = {
                "PMID": pmid_elem.text if pmid_elem is not None else None,
                "ArticleTitle": title_text,
                "AbstractText": abstract_text,
                "Journal": journal_elem.text if journal_elem is not None else None,
                "YearPublished": date_elem.text if date_elem is not None else None
            }
            all_cancer_metadata.append(metadata)
            cancer_pmids.append(metadata['PMID'])
        except Exception as e:
            pmid_text = pmid_elem.text if pmid_elem is not None else "UNKNOWN"
            print(f"Error parsing article PMID {pmid_text}: {e}")
            failed_pmids.append(pmid_text)
            continue

    time.sleep(1)

print(f"Retrieved metadata for {len(all_cancer_metadata)} Cancer articles.")
print(f"Failed PMIDs in first pass: {len(failed_pmids)}")


Retrieved metadata for 1000 Cancer articles.
Failed PMIDs in first pass: 0


In [34]:
for paper in all_alz_metadata:
    paper["query"] = "alzheimers"

for paper in all_cancer_metadata:
    paper["query"] = "cancer"

In [ ]:
# make the metadata json - not needed. delete if rerun and it works
# all_paper_metadata = all_alz_metadata + all_cancer_metadata
# print(all_paper_metadata[:500])
# print(type(all_paper_metadata))


[{'PMID': '41058862', 'ArticleTitle': "Deep learning assessment of disproportionately enlarged subarachnoid-space hydrocephalus in Hakim's disease or idiopathic normal pressure hydrocephalus.", 'AbstractText': "Disproportionately enlarged subarachnoid-space hydrocephalus (DESH) is a key feature of Hakim's disease (synonymous with idiopathic normal pressure hydrocephalus; iNPH). However, it previously had been only subjectively evaluated. This study aims to evaluate the usefulness of MRI indices, derived from deep learning segmentation of cerebrospinal fluid (CSF) spaces, for DESH detection and to establish their optimal thresholds. This study retrospectively enrolled a total of 1009 participants, including 77 patients diagnosed with Hakim's disease, 380 healthy volunteers, 163 with mild cognitive impairment, 256 with Alzheimer's disease, and 217 with other types of neurodegenerative diseases. DESH, ventriculomegaly, tightened sulci in the high convexities, and Sylvian fissure dilatatio

In [ ]:
# ensure none are missing and everything was pulled over correctly - helpful when troubleshooting
# not needed if don't need to tuen into  json

# Missing titles
missing_titles = [p for p in all_paper_metadata if not p.get("ArticleTitle")]

# Missing abstracts
missing_abstracts = [p for p in all_paper_metadata if not p.get("AbstractText")]

# Missing PMIDs
missing_pmids = [p for p in all_paper_metadata if not p.get("PMID")]

print(f"Missing titles: {len(missing_titles)}")
print(f"Missing abstracts: {len(missing_abstracts)}")
print(f"Missing PMIDs: {len(missing_pmids)}")


Missing titles: 0
Missing abstracts: 101
Missing PMIDs: 0


[]

In [44]:
# convert list of all papers metadata into paper dictionary

papers = {
    paper["PMID"]: {
        "ArticleTitle": paper.get("ArticleTitle", ""),
        "AbstractText": paper.get("AbstractText", ""),
        "query": paper.get('query', "")
    }
    for paper in all_paper_metadata
    if paper.get("PMID")  # only include valid PMIDs
}

print(f"Prepared {len(papers)} papers for embedding generation.") # ensure all papers were processed

Prepared 1996 papers for embedding generation.


In [ ]:
# technically not needed anymore, was helpful for troubleshooting when weren't all pulling

missing_title = [p for p in papers.values() if not p.get("ArticleTitle")]
missing_abs = [p for p in papers.values() if not p.get("AbstractText")]

print(f"Missing titles: {len(missing_title)}")
print(f"Missing abstracts: {len(missing_abs)}")

Missing titles: 0
Missing abstracts: 101


In [39]:
# used ChatGPT to add the get_abstract function because error threw first time

# we can use a persistent dictionary (via shelve) so we can stop and restart if needed
# alternatively, do the same but with embeddings starting as an empty dictionary
def get_abstract(paper):
    return paper.get("AbstractText", "") or ""

embeddings = {}
for pmid, paper in tqdm.tqdm(papers.items()):
    data = [paper["ArticleTitle"] + tokenizer.sep_token + get_abstract(paper)]
    inputs = tokenizer(
        data, padding=True, truncation=True, return_tensors="pt", max_length=512
    )
    result = model(**inputs)
    # take the first token in the batch as the embedding
    embeddings[pmid] = result.last_hidden_state[:, 0, :].detach().numpy()[0]

# turn our dictionary into a list
embeddings = [embeddings[pmid] for pmid in papers.keys()]

100%|██████████| 1996/1996 [19:17<00:00,  1.72it/s]  


In [45]:
pca = decomposition.PCA(n_components=3)
embeddings_pca = pd.DataFrame(
    pca.fit_transform(embeddings),
    columns=['PC0', 'PC1', 'PC2']
)
embeddings_pca["query"] = [paper["query"] for paper in papers.values()]

In [ ]:
# https://plotly.com/python/pca-visualization/

labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

color_map = {
    "alzheimers": "#1f77b4",
    "cancer": "#d62728"
}

fig = px.scatter_matrix(
    embeddings_pca,
    labels=labels,
    dimensions=['PC0', 'PC1', 'PC2'],
    color='query',
    color_discrete_map=color_map
)

fig.update_traces(diagonal_visible=False)
fig.show()

In [60]:
# I did the above graphs and because they have extra comparisons, I asked ChatGPT to refine to only the needed analysis 

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Define color map
color_map = {
    "alzheimers": "#1f77b4",
    "cancer": "#d62728"
}

# Define variance labels for axes
labels = {
    f"PC{i}": f"PC{i} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

# Create subplots — 1 row, 3 columns
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "PC0 vs PC1", "PC0 vs PC2", "PC1 vs PC2"
))

# --- Plot 1: PC0 vs PC1 ---
for query, color in color_map.items():
    df = embeddings_pca[embeddings_pca["query"] == query]
    fig.add_trace(
        go.Scatter(
            x=df["PC0"], y=df["PC1"],
            mode='markers',
            name=query,
            marker=dict(color=color, size=6, opacity=0.7)
        ),
        row=1, col=1
    )

# --- Plot 2: PC0 vs PC2 ---
for query, color in color_map.items():
    df = embeddings_pca[embeddings_pca["query"] == query]
    fig.add_trace(
        go.Scatter(
            x=df["PC0"], y=df["PC2"],
            mode='markers',
            name=query,
            marker=dict(color=color, size=6, opacity=0.7),
            showlegend=False  # avoid duplicate legends
        ),
        row=1, col=2
    )

# --- Plot 3: PC1 vs PC2 ---
for query, color in color_map.items():
    df = embeddings_pca[embeddings_pca["query"] == query]
    fig.add_trace(
        go.Scatter(
            x=df["PC1"], y=df["PC2"],
            mode='markers',
            name=query,
            marker=dict(color=color, size=6, opacity=0.7),
            showlegend=False
        ),
        row=1, col=3
    )

# Layout
fig.update_layout(
    height=500,
    width=1200,
    title_text="2D PCA Scatter Plots by Query Type",
    template="plotly_white"
)

# Axis labels
fig.update_xaxes(title_text=labels["PC0"], row=1, col=1)
fig.update_yaxes(title_text=labels["PC1"], row=1, col=1)

fig.update_xaxes(title_text=labels["PC0"], row=1, col=2)
fig.update_yaxes(title_text=labels["PC2"], row=1, col=2)

fig.update_xaxes(title_text=labels["PC1"], row=1, col=3)
fig.update_yaxes(title_text=labels["PC2"], row=1, col=3)

fig.write_image('PCA_analysis.png')
fig.show()


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
